## Assignment 1: Prompting for Program Repair

In this assignment, you will design prompts to repair buggy Python functions and evaluate the quality of the bug repair using unit tests.

What you will do:
- Use this notebook as a guide to complete your assignment.
- Inside the `data` directory, you’ll find one folder per buggy Python function (`data/BUGxxx/`), each with example tests in `test_bug.py`.
- Experiment with prompting techniques using the model (Gemma 3 270M IT), and observe how variations in prompt wording affect bug repair performance.
- For each buggy Python function, record the prompts you used, then choose one final repaired code to submit.

Submission Instructions:  
Submit a ZIP file of the `results` folder, containing the two items listed below, named `SWE270P_A1.ZIP`.  
**Please make sure to strictly follow the submission format and filename, as an autograder will be used for grading. <br> An incorrect format or filename may result in a score of 0.**  
 1) `prompts.jsonl`: all the prompts you used. This notebook includes a helper function that logs them to `results/prompts.jsonl`.
 2) `BUGxxx.py` files: one final repaired code file per buggy Python function (e.g., `BUG001.py`), saved under `results/final_patches/`.  


Required File Structure:  
Your submission ZIP file **must** follow this exact file structure for compatibility with the autograder:    
```
SWE270P_A1.ZIP
└── results
    ├── prompts.jsonl
    └── final_patches
        ├── BUG000.py
        ├── ...
        └── BUG009.py
```

Helpful Guidelines:
- Use `test_bug.py` to evaluate the quality of repaired code.
- Add your own test cases to check the robustness of your repaired code.
- Final submissions will be graded using an autograder. Do not modify any function names or parameter definitions.
- Each final patch must be a complete replacement for that bug's `buggy.py`. Keep its imports and helper functions, not only the fixed function (e.g., BUG001 needs `get` and `insert_or_update`).

Recommended process:
1) Work on one bug at a time (e.g., BUG001).
2) Select a prompting technique (e.g., zero-shot, chain-of-thought, tests-aware, or self-refine).
3) Using the selected technique, draft your prompt and call the model.
4) Copy the repaired code into that bug's `buggy.py` and run its tests with `test_bug.py` (e.g., in `data/BUG001/`). <br>
    **For example**, for BUG001, run `pytest test_bug.py` in a terminal inside `data/BUG001/`, or run `!cd data/BUG001 && pytest test_bug.py` in a notebook cell.
5) Modify the prompt using different wording and structure, or try another prompting technique until the repaired code passes all tests.   
6) Log the prompting technique and the prompt used to call the model in `prompts.jsonl`.
7) Save your final repaired code for each bug (e.g., `BUG001.py`) for submission.



## 0. Setup (Required: Gemma 3 270M IT)

This assignment **must use** `google/gemma-3-270m-it` via Hugging Face Transformers (local CPU/GPU). No external APIs are needed.

We recommend using Google Colab. If you're using Colab, make sure you have a GPU runtime (Runtime → Change runtime type → select T4 GPU). If you run locally, use Python 3.10 or newer.

### Completed local environment

This solution uses the Python 3.12 virtual environment at `../.venv`. From this directory, run:

```bash
python3.12 -m venv ../.venv  # Only when creating a fresh environment.
../.venv/bin/python -m pip install -r requirements.txt
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 ../.venv/bin/python run_experiments.py
../.venv/bin/python evaluate.py
../.venv/bin/python -m pytest tests -q
../.venv/bin/python build_submission.py
```

The completed experiment uses cached model weights and local CPU inference. All assignment text, prompts, code documentation, and results are in English.


In [1]:
import sys
from pathlib import Path

assert sys.prefix != sys.base_prefix, "Select the assignment's venv kernel."
print("Python executable:", sys.executable)
print("Virtual environment:", sys.prefix)

Python executable: /Users/ericsong/Courses/SWE270P/assignment/assignment1/SWE270P_A1/../.venv/bin/python
Virtual environment: /Users/ericsong/Courses/SWE270P/assignment/assignment1/.venv


In [2]:
from importlib.metadata import version

for package in ("torch", "transformers", "accelerate", "pytest", "huggingface-hub"):
    print(f"{package}=={version(package)}")

torch==2.14.1
transformers==5.20.2
accelerate==1.15.0
pytest==9.1.1
huggingface-hub==2.2.0


### Model access

The completed run used the cached `google/gemma-3-270m-it` snapshot. No token is stored in this project. On a fresh machine, accept access on the [official model page](https://huggingface.co/google/gemma-3-270m-it), authenticate locally with `hf auth login`, and download the model with `hf download google/gemma-3-270m-it` before running the offline experiment.

The cells below replay the **actual saved prompts and raw completions**, so re-running this notebook does not silently issue duplicate model calls. Use `run_experiments.py` for inference; every attempt is logged before code extraction or testing. See `repair_workflow.py` for the inference, extraction, logging, and patch-saving implementations.

In [3]:
from huggingface_hub import try_to_load_from_cache

cached_config = try_to_load_from_cache("google/gemma-3-270m-it", "config.json")
assert isinstance(cached_config, str), "Download the required model before inference."
print("Required model configuration is cached locally.")

Required model configuration is cached locally.


In [4]:
from repair_workflow import MODEL_ID, generate_text, load_model

# generate_text applies the tokenizer's chat template, uses deterministic
# inference, and decodes only newly generated tokens. See repair_workflow.py.
# Use run_prompt rather than generate_text directly for any NEW experiment,
# so the exact prompt and raw completion are recorded automatically.
print("Required inference model:", MODEL_ID)

Required inference model: google/gemma-3-270m-it


## Prompting Techniques: Zero‑Shot, One‑Shot, Chain‑of‑Thought, and Few‑Shot  
**Goal:** Practice different prompting strategies and compare their effects on bug repair correctness.

After generating the fix for a problem, you can paste it into the corresponding `buggy.py` and run `pytest` for that folder to see if it passes.

In [5]:
import json
from repair_workflow import (
    ROOT, PROMPTS_PATH, PATCH_DIR, extract_code, log_prompt,
    run_prompt, save_final_patch, validate_interface,
)
from prompt_designs import TECHNIQUES, make_prompt

records = [json.loads(line) for line in PROMPTS_PATH.read_text().splitlines()]
by_attempt = {r["attempt_id"]: r for r in records if "attempt_id" in r}

def recorded_output(bug, technique, variant):
    """Replay an actual inference result, without changing the append-only log."""
    attempt_id = f"{bug}_{technique}_{variant + 1}"
    record = by_attempt[attempt_id]
    assert record["prompt"] == make_prompt(bug, technique, variant)
    assert record["model"] == MODEL_ID
    return record["completion"]

print(f"Loaded {len(by_attempt)} target attempts and {len(records) - len(by_attempt)} setup attempts.")

Loaded 91 target attempts and 1 setup attempts.


## 1. Zero‑Shot
A prompting technique that asks the model to solve the task **without** examples. Include specific output constraints.

In [6]:
example_zero_shot_prompt = make_prompt("BUG000", "zero-shot", 0)
example_zero_shot_output = recorded_output("BUG000", "zero-shot", 0)
print("----- ZERO-SHOT: CONTRACT-FIRST REPAIR -----\n", example_zero_shot_prompt)
print("----- RAW GEMMA OUTPUT -----\n", example_zero_shot_output)

----- ZERO-SHOT: CONTRACT-FIRST REPAIR -----
 Repair this Python module from its behavioral contract. Inspect the implementation for all defects.
Target file: BUG000/buggy.py
Contract: Return whether two DISTINCT list positions have absolute difference strictly less than threshold. Do not mutate numbers. Empty/singleton lists return False; equal distance is not close.
Original complete module:
```python
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = elem - elem2
                if distance < threshold:
                    return True

    return Fals

### Logging and final-patch helpers

`run_prompt(technique, prompt, bug_file)` calls Gemma and immediately appends the exact prompt and unedited completion to `results/prompts.jsonl`. The experiment runner additionally records a unique attempt ID and generation settings.

`extract_code(response)` accepts a fenced Python module or plain Python and rejects malformed code. `save_final_patch("BUG000.py", response)` checks every original function signature before saving a complete replacement module. Tests must still be run afterward. Invalid placeholder filenames are rejected.

Raw outputs, test outcomes, and final reviewed patches are separate artifacts. Failed outputs remain in the log. Final edits are documented in `experiments/final_patch_provenance.json` and `REPORT.md`.

In [7]:
your_zero_shot_prompt = make_prompt("BUG000", "zero-shot", 1)
your_zero_shot_output = recorded_output("BUG000", "zero-shot", 1)
print("----- ZERO-SHOT: MAINTAINER REVIEW -----\n", your_zero_shot_prompt)
print("----- RAW GEMMA OUTPUT -----\n", your_zero_shot_output)

----- ZERO-SHOT: MAINTAINER REVIEW -----
 Act as a maintainer reviewing a bug fix. Make a focused repair, preserving the public interface. Check empty inputs, termination, and input mutation against the contract.
Target file: BUG000/buggy.py
Contract: Return whether two DISTINCT list positions have absolute difference strictly less than threshold. Do not mutate numbers. Empty/singleton lists return False; equal distance is not close.
Original complete module:
```python
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = elem - elem2
                if dis

## 2. One‑Shot
A prompting technique that includes **one worked example** before the target task.

In [8]:
prompt = make_prompt("BUG007", "one-shot", 0)
completion = recorded_output("BUG007", "one-shot", 0)
print("----- ONE-SHOT: POINTER-STATE EXAMPLE -----\n", prompt)
print("----- RAW GEMMA OUTPUT -----\n", completion)

----- ONE-SHOT: POINTER-STATE EXAMPLE -----
 Follow the repair pattern in this single worked example, then fix the target.
Worked repair 1: Preserve the current node before advancing.
Buggy:
```python
def advance(previous, current):
    current = current.successor
    return previous, current
```
Corrected:
```python
def advance(previous, current):
    previous, current = current, current.successor
    return previous, current
```
Target file: BUG007/buggy.py
Contract: Reverse an acyclic singly linked list IN PLACE using .successor. Return the original tail as the new head; original head becomes the tail with successor None. Preserve node identities and handle None and a singleton.
Original complete module:
```python
def reverse_linked_list(node):
    prevnode = None
    while node:
        nextnode = node.successor
        node.successor = prevnode
        node = nextnode
    return prevnode


"""
Reverse Linked List

Reverses a linked list and returns the new head.

Input:
    node: 

In [9]:
prompt = make_prompt("BUG007", "one-shot", 1)
completion = recorded_output("BUG007", "one-shot", 1)
print("----- ONE-SHOT: NULL-GUARD EXAMPLE -----\n", prompt)
print("----- RAW GEMMA OUTPUT -----\n", completion)

----- ONE-SHOT: NULL-GUARD EXAMPLE -----
 Use this single before/after review as a guide to preserving the invariant in the target.
Worked repair 1: Guard both dereferences.
Buggy:
```python
def two_steps(node):
    return node.successor.successor
```
Corrected:
```python
def two_steps(node):
    if node is None or node.successor is None:
        return None
    return node.successor.successor
```
Target file: BUG007/buggy.py
Contract: Reverse an acyclic singly linked list IN PLACE using .successor. Return the original tail as the new head; original head becomes the tail with successor None. Preserve node identities and handle None and a singleton.
Original complete module:
```python
def reverse_linked_list(node):
    prevnode = None
    while node:
        nextnode = node.successor
        node.successor = prevnode
        node = nextnode
    return prevnode


"""
Reverse Linked List

Reverses a linked list and returns the new head.

Input:
    node: The head of a singly-linked list



## 3. Chain‑of‑Thought (CoT)
A prompting technique that asks the model for a brief high‑level plan (step-by-step) before the final code.

In [10]:
prompt = make_prompt("BUG008", "chain-of-thought", 0)
completion = recorded_output("BUG008", "chain-of-thought", 0)
print("----- CHAIN-OF-THOUGHT: INVARIANT SUMMARY -----\n", prompt)
print("----- RAW GEMMA OUTPUT -----\n", completion)

----- CHAIN-OF-THOUGHT: INVARIANT SUMMARY -----
 Review this program repair task.
Target file: BUG008/buggy.py
Contract: Use Floyd's tortoise-and-hare algorithm on .successor: move the slow pointer once and the fast pointer twice. Return a Boolean for empty, finite, self-cycle, and tail-into-cycle lists. Check that the fast pointer and its next node exist before advancing twice. Do not mutate links.
Original complete module:
```python
def detect_cycle(node):
    hare = tortoise = node

    while node:
        if hare is None:
            return False

        tortoise = tortoise.successor
        hare = hare.successor

        if hare is tortoise:
            return True



"""
Linked List Cycle Detection
tortoise-hare

Implements the tortoise-and-hare method of cycle detection.

Input:
    node: The head node of a linked list

Output:
    Whether the linked list is cyclic
"""
```
Before the code, give 2-3 brief high-level bullets naming the violated invariant, the correction, and a bo

In [11]:
prompt = make_prompt("BUG008", "chain-of-thought", 1)
completion = recorded_output("BUG008", "chain-of-thought", 1)
print("----- CHAIN-OF-THOUGHT: COUNTEREXAMPLE SUMMARY -----\n", prompt)
print("----- RAW GEMMA OUTPUT -----\n", completion)

----- CHAIN-OF-THOUGHT: COUNTEREXAMPLE SUMMARY -----
 Review this program repair task.
Target file: BUG008/buggy.py
Contract: Use Floyd's tortoise-and-hare algorithm on .successor: move the slow pointer once and the fast pointer twice. Return a Boolean for empty, finite, self-cycle, and tail-into-cycle lists. Check that the fast pointer and its next node exist before advancing twice. Do not mutate links.
Original complete module:
```python
def detect_cycle(node):
    hare = tortoise = node

    while node:
        if hare is None:
            return False

        tortoise = tortoise.successor
        hare = hare.successor

        if hare is tortoise:
            return True



"""
Linked List Cycle Detection
tortoise-hare

Implements the tortoise-and-hare method of cycle detection.

Input:
    node: The head node of a linked list

Output:
    Whether the linked list is cyclic
"""
```
Counterexample to check: A two-node finite list must return False; a one-node self-loop must return T

## 4. Few‑Shot
A prompting technique that provides multiple examples to shape the model's behavior.

In [12]:
prompt = make_prompt("BUG009", "few-shot", 0)
completion = recorded_output("BUG009", "few-shot", 0)
print("----- FEW-SHOT: TWO RELATED REPAIRS -----\n", prompt)
print("----- RAW GEMMA OUTPUT -----\n", completion)

----- FEW-SHOT: TWO RELATED REPAIRS -----
 Generalize the two worked repairs to the target module.
Worked repair 1: Bounds and content checks must both hold.
Buggy:
```python
def opening_at(s, i):
    return i >= 0 or s[i] == '('
```
Corrected:
```python
def opening_at(s, i):
    return i >= 0 and s[i] == '('
```

Worked repair 2: Count characters, not pairs.
Buggy:
```python
def add_pair(length):
    return length + 1
```
Corrected:
```python
def add_pair(length):
    return length + 2
```
Target file: BUG009/buggy.py
Contract: Return the CHARACTER LENGTH of the longest contiguous well-formed parentheses substring. Empty/unmatched inputs return zero. A matching pair adds TWO characters. Check the matching-open index before indexing and require both a valid index AND an opening parenthesis. Preserve the name longestValidParentheses(s).
Original complete module:
```python
def longestValidParentheses(s):
    maxans = 0
    dp = [0] * len(s)
    for i in range(1, len(s)):
        if s[i] 

In [13]:
prompt = make_prompt("BUG009", "few-shot", 1)
completion = recorded_output("BUG009", "few-shot", 1)
print("----- FEW-SHOT: THREE REPAIR REVIEWS -----\n", prompt)
print("----- RAW GEMMA OUTPUT -----\n", completion)

----- FEW-SHOT: THREE REPAIR REVIEWS -----
 Use these three small repair reviews as a checklist of state, bounds, and progress.
Worked repair 1: Bounds and content checks must both hold.
Buggy:
```python
def opening_at(s, i):
    return i >= 0 or s[i] == '('
```
Corrected:
```python
def opening_at(s, i):
    return i >= 0 and s[i] == '('
```

Worked repair 2: Count characters, not pairs.
Buggy:
```python
def add_pair(length):
    return length + 1
```
Corrected:
```python
def add_pair(length):
    return length + 2
```

Worked repair 3: Guard both dereferences.
Buggy:
```python
def two_steps(node):
    return node.successor.successor
```
Corrected:
```python
def two_steps(node):
    if node is None or node.successor is None:
        return None
    return node.successor.successor
```
Target file: BUG009/buggy.py
Contract: Return the CHARACTER LENGTH of the longest contiguous well-formed parentheses substring. Empty/unmatched inputs return zero. A matching pair adds TWO characters. Chec

## Submission checklist

Before you submit:
- Ensure `results/prompts.jsonl` contains all prompts you tried (one JSON line per attempt). <br> FYI: Make sure to check the rubric for full marks on this!
- Ensure you have one final repaired code for each buggy Python function, saved under `results/final_patches/`.
- Make sure each file is named exactly `BUGxxx.py` (e.g., `BUG001.py`).

What to upload:
- `results/prompts.jsonl`
- `results/final_patches/` folder containing all 10 repaired code files, one per buggy Python function

**Required File Structure**:  
Your submission ZIP file **must** follow this exact file structure for compatibility with the autograder:    
```
SWE270P_A1.ZIP
└── results
    ├── prompts.jsonl
    └── final_patches
        ├── BUG000.py
        ├── ...
        └── BUG009.py
```

## Experiment comparison and final validation

The experiment covers every combination of ten target bugs, four required methods, and two prompt designs (80 target calls). The setup smoke test is also retained in the prompt log. Generation uses the required model, CPU float32, greedy decoding, and a 768-token output limit for primary attempts; failure-guided revisions allow up to 1024 new tokens.

Prompt variants deliberately differ in their repair strategy: contract-first versus maintainer review, two different single worked examples, invariant summary versus a concrete counterexample, and two related examples versus three repair reviews.

The following table reports **raw Gemma candidate** performance. Passing the three supplied tests is not proof of general correctness. Final patches received additional review and corrections and were checked against the independent robustness suite. Their results are reported separately; they are not presented as unedited Gemma outputs.

The ten failure-guided revisions are recorded separately in `experiments/refinement_evaluations.jsonl`. BUG000 uses a passing unedited Gemma revision; the other final files include assistant corrections, as documented in the provenance manifest.

A later review re-runs the successful BUG000 prompt to check deterministic local inference. That fresh prompt and raw completion are also logged; it is not counted as a new primary prompt design.

In [14]:
from collections import Counter

expected = {
    f"BUG{i:03d}_{method}_{variant}"
    for i in range(10) for method in TECHNIQUES for variant in (1, 2)
}
assert expected <= by_attempt.keys()
assessments = [json.loads(line) for line in (ROOT / "experiments/candidate_evaluations.jsonl").read_text().splitlines()]
print(f"{'Method':<20} {'Attempts':>8} {'All supplied tests + interface':>31}")
for method in TECHNIQUES:
    matching = [a for a in assessments if a["technique"] == method]
    passed = sum(a["status"] == "passed" for a in matching)
    print(f"{method:<20} {len(matching):>8} {passed:>31}")

final_results = json.loads((ROOT / "experiments/final_tests.json").read_text())
assert all(result["all_passed"] for result in final_results.values())
for i in range(10):
    bug_file = f"BUG{i:03d}.py"
    code = (PATCH_DIR / bug_file).read_text()
    validate_interface(bug_file, code)
    compile(code, bug_file, "exec")
print("Final reviewed patches: all 30 supplied tests passed.")
print("Additional validation and limitations: see REPORT.md.")
print("Submission archive:", ROOT / "SWE270P_A1.ZIP")
refinements = [json.loads(line) for line in (ROOT / "experiments/refinement_evaluations.jsonl").read_text().splitlines()]
print("Failure-guided revisions passing supplied tests:", sum(r["status"] == "passed" for r in refinements), "/", len(refinements))
import xml.etree.ElementTree as ET
robustness = list(ET.parse(ROOT / "experiments/robustness_tests.xml").iter("testcase"))
assert all(c.find("failure") is None and c.find("error") is None for c in robustness)
print("Additional robustness cases passed:", len(robustness))

Method               Attempts  All supplied tests + interface
zero-shot                  20                               0
one-shot                   20                               0
chain-of-thought           20                               0
few-shot                   20                               0
Final reviewed patches: all 30 supplied tests passed.
Additional validation and limitations: see REPORT.md.
Submission archive: /Users/ericsong/Courses/SWE270P/assignment/assignment1/SWE270P_A1/SWE270P_A1.ZIP
Failure-guided revisions passing supplied tests: 1 / 10
Additional robustness cases passed: 109
